# AnRAG mixed benchmark on Colab\n\nThis notebook runs the primary retrieval benchmark with **FiQA replaced by RAGBench**.\n\nBenchmark mix:\n- HotpotQA: multi-hop retrieval\n- SciFact: evidence retrieval\n- RAGBench: real-world RAG retrieval\n\nThe run is retrieval-only: no Ollama, no OCR, no VLM, no answer generation.

In [ ]:
# Configuration\nfrom pathlib import Path\n\nREPO_URL = "https://github.com/c1phanthanh75-ship-it/AnRAG.git"\nBRANCH = "main"\nREPO_DIR = Path("/content/AnRAG")\nWORK = Path("/content/anrag_mix_bench")\nWORK.mkdir(parents=True, exist_ok=True)\n\n# Keep these modest for Colab smoke runs. Increase when the pipeline is stable.\nN_HOTPOT = 200\nN_SCIFACT = 200\nN_RAGBENCH = 200\nMAX_SCIFACT_DOCS = 1200\n\nRAGBENCH_SUBSET = "covidqa"\nRAGBENCH_SPLIT = "test"\n\nTOP_K = 8\nBUDGET_TOKENS = 1200\nRANDOM_SEED = 42\n\nWEIGHTS = {"HotpotQA": 0.50, "SciFact": 0.25, "RAGBench": 0.25}\nprint("Work dir:", WORK)

In [ ]:
# Clone and install AnRAG\nimport os\n\nif not REPO_DIR.exists():\n    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}\nelse:\n    print("Repo already exists:", REPO_DIR)\n\nos.chdir(REPO_DIR)\nprint("cwd =", Path.cwd())\n\n!python -m pip install -q -U pip\n!python -m pip install -q -r requirements.txt\n!python -m pip install -q -e .\n!python -m pip install -q datasets beir sentence-transformers pandas matplotlib tqdm

In [ ]:
# Verify this checkout supports RAGBench.\n# If this assertion fails, pull the repo version that contains the RAGBench loader.\nfrom anrag.benchmark_gt import detect_benchmark_format, load_ragbench\n\nprint("RAGBench loader is available:", load_ragbench.__name__)

In [ ]:
# Helpers\nimport json\nimport random\nfrom collections.abc import Mapping, Sequence\n\nrandom.seed(RANDOM_SEED)\n\ndef to_jsonable(value):\n    if isinstance(value, Mapping):\n        return {str(k): to_jsonable(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [to_jsonable(v) for v in value]\n    if hasattr(value, "item"):\n        try:\n            return value.item()\n        except Exception:\n            pass\n    return value\n\ndef write_jsonl(path, rows):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as handle:\n        for row in rows:\n            handle.write(json.dumps(to_jsonable(row), ensure_ascii=False) + "\\n")\n    print(path, "rows=", len(rows))\n    return path\n\ndef load_report_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))

In [ ]:
# Build HotpotQA subset as raw HotpotQA JSONL\nfrom datasets import load_dataset\n\nhotpot = list(load_dataset("hotpotqa/hotpot_qa", name="distractor", split="validation"))\nhotpot_sample = random.sample(hotpot, min(N_HOTPOT, len(hotpot)))\nhotpot_path = write_jsonl(WORK / "hotpotqa_dev_subset.jsonl", hotpot_sample)\n\nprint("HotpotQA sample question:", hotpot_sample[0]["question"])

In [ ]:
# Build SciFact subset in BeIR layout\nfrom beir import util\nfrom beir.datasets.data_loader import GenericDataLoader\n\nbeir_root = WORK / "beir"\nbeir_root.mkdir(parents=True, exist_ok=True)\nscifact_path = Path(util.download_and_unzip(\n    "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip",\n    str(beir_root),\n))\n\nscifact_corpus, scifact_queries, scifact_qrels = GenericDataLoader(str(scifact_path)).load(split="test")\nquery_ids = [qid for qid in scifact_queries.keys() if scifact_qrels.get(qid)][:N_SCIFACT]\n\nneeded_docs = set()\nfor qid in query_ids:\n    needed_docs.update(scifact_qrels[qid].keys())\n\nnegative_candidates = [doc_id for doc_id in scifact_corpus.keys() if doc_id not in needed_docs]\nnegative_count = max(0, min(MAX_SCIFACT_DOCS - len(needed_docs), len(negative_candidates)))\nselected_docs = set(needed_docs) | set(random.sample(negative_candidates, negative_count))\n\nmini_scifact = WORK / "scifact_beir_subset"\n(mini_scifact / "qrels").mkdir(parents=True, exist_ok=True)\n\nwrite_jsonl(\n    mini_scifact / "corpus.jsonl",\n    [\n        {"_id": doc_id, "title": scifact_corpus[doc_id].get("title", ""), "text": scifact_corpus[doc_id].get("text", "")}\n        for doc_id in sorted(selected_docs)\n    ],\n)\nwrite_jsonl(\n    mini_scifact / "queries.jsonl",\n    [{"_id": qid, "text": scifact_queries[qid]} for qid in query_ids],\n)\nwith (mini_scifact / "qrels" / "test.tsv").open("w", encoding="utf-8") as handle:\n    for qid in query_ids:\n        for doc_id, score in scifact_qrels[qid].items():\n            if doc_id in selected_docs:\n                handle.write(f"{qid}\\t{doc_id}\\t{int(score)}\\n")\n\nprint("SciFact queries:", len(query_ids))\nprint("SciFact docs:", len(selected_docs))

In [ ]:
# Build RAGBench subset. This replaces FiQA in the mixed benchmark.\nragbench = load_dataset("galileo-ai/ragbench", RAGBENCH_SUBSET, split=RAGBENCH_SPLIT)\nragbench_rows = [to_jsonable(row) for row in ragbench.select(range(min(N_RAGBENCH, len(ragbench))))]\nragbench_path = write_jsonl(WORK / f"ragbench_{RAGBENCH_SUBSET}_{RAGBENCH_SPLIT}_subset.jsonl", ragbench_rows)\n\nprint("RAGBench subset:", RAGBENCH_SUBSET, RAGBENCH_SPLIT)\nprint("RAGBench sample question:", ragbench_rows[0]["question"])\nprint("Relevant sentence keys:", ragbench_rows[0].get("all_relevant_sentence_keys"))

In [ ]:
# Run ablations\nfrom anrag.ablation import run_ablations\n\nreports = {}\n\nreports["HotpotQA"] = run_ablations(\n    hotpot_path,\n    benchmark_format="hotpotqa",\n    top_k=TOP_K,\n    budget_tokens=BUDGET_TOKENS,\n)\n\nreports["SciFact"] = run_ablations(\n    mini_scifact,\n    benchmark_format="beir",\n    top_k=TOP_K,\n    budget_tokens=BUDGET_TOKENS,\n)\n\nreports["RAGBench"] = run_ablations(\n    ragbench_path,\n    benchmark_format="ragbench",\n    top_k=TOP_K,\n    budget_tokens=BUDGET_TOKENS,\n)\n\nprint("Done")

In [ ]:
# Summarize results\nimport pandas as pd\n\nrows = []\nfor name, report in reports.items():\n    scores = {score.key: score for score in report.scores}\n    baseline = scores["baseline"].metrics\n    anchor_only = scores["anchor_only"].metrics\n    full = scores["full"].metrics\n    rows.append({\n        "Dataset": name,\n        "Weight": WEIGHTS[name],\n        "Plain RAG R@K": baseline.recall_at_k,\n        "Anchor Only R@K": anchor_only.recall_at_k,\n        "AnRAG Full R@K": full.recall_at_k,\n        "Delta Full-Baseline": round(full.recall_at_k - baseline.recall_at_k, 4),\n        "Zero Gold": scores["full"].gold_sanity.zero_gold_count,\n        "Avg Gold Size": scores["full"].gold_sanity.avg_gold_size,\n    })\n\ndf = pd.DataFrame(rows)\ndf["Weighted Full"] = df["Weight"] * df["AnRAG Full R@K"]\ndisplay(df)\nprint("FINAL MIXED SCORE =", round(df["Weighted Full"].sum(), 4))

In [ ]:
# Save reports for later inspection\nreports_dir = WORK / "reports"\nreports_dir.mkdir(parents=True, exist_ok=True)\n\nfor name, report in reports.items():\n    out = reports_dir / f"{name.lower()}_report.json"\n    out.write_text(json.dumps(report.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")\n    print(out)\n\ndf.to_csv(reports_dir / "mixed_summary.csv", index=False)\nprint("Saved summary:", reports_dir / "mixed_summary.csv")

In [ ]:
# Optional: download report files from Colab\nimport shutil\n\narchive = shutil.make_archive(str(WORK / "anrag_ragbench_reports"), "zip", reports_dir)\nprint("Archive:", archive)\n\ntry:\n    from google.colab import files\n    files.download(archive)\nexcept Exception as exc:\n    print("Download skipped:", exc)